# Phase 15 — Reviewer evidence map
**Owner: Shafin.** This notebook reads the copied review letter and submitted manuscript as evidence, not as instructions or authorization. It creates a traceable point-by-point map, exact file paths and hashes, and suggested narrower claims. It does not edit the manuscript, send a journal response or verify journal portal status.

Run in **enhancer**. Run notebooks 13 and 14 first for the fullest evidence; if their final summaries are missing, this notebook preserves an explicit pending status instead of fabricating results. “Addressed” means this concern has a completed technical evidence package within the stated new experiment; it does not mean the editor has accepted the response or the old manuscript has been repaired.

In [1]:
from pathlib import Path
import os, sys, io, json, zipfile, hashlib, itertools
import numpy as np
import pandas as pd
ROOT=Path(r"D:\Aditta Thesis\For Shafin\Shafin's Contribution").resolve()
assert ROOT.is_dir() and ROOT.name=="Shafin's Contribution"
assert Path(sys.executable).resolve()==Path(r'C:\Users\User\.conda\envs\enhancer\python.exe').resolve(), 'Select the enhancer Anaconda kernel.'
OUT=ROOT/'phase15_outputs';OUT.mkdir(exist_ok=True)
P12=ROOT/'phase12_outputs/enhancer_phase12_work'
def digest(raw):return hashlib.sha256(raw).hexdigest()
def filehash(p,algorithm='sha256'):
 with Path(p).open('rb') as f:return hashlib.file_digest(f,algorithm).hexdigest()
def member(name):
 found=[]
 for folder in ['phase12_inputs','reference_inputs']:
  for p in (ROOT/folder).glob('*.zip'):
   with zipfile.ZipFile(p) as z:
    for m in z.namelist():
     if m.split('/')[-1]==name:found.append(z.read(m))
 assert found, 'Missing copied input: '+name
 assert len({digest(b) for b in found})==1, 'Ambiguous input versions: '+name
 return found[0]
def csvmember(name):return pd.read_csv(io.BytesIO(member(name)))
def dump_json(name,obj):(OUT/name).write_text(json.dumps(obj,indent=2,sort_keys=True,allow_nan=False)+'\n',encoding='utf-8')
def write_table(name,frame):frame.to_csv(OUT/name,index=False,float_format='%.12g')
prov=json.loads((P12/'results/phase12_provenance.json').read_text())
assert prov['smoke_test'] is False
for name,h in prov['config']['input_sha256'].items():assert digest(member(name))==h,name
full=csvmember('phase9_full_activity_all_cells.csv')
splits=csvmember('phase9_frozen_splits.csv')
assert full.record_id.is_unique and splits.record_id.is_unique
DATA=full.merge(splits.drop(columns=[c for c in splits if c in full and c!='record_id']),on='record_id',validate='one_to_one')
assert len(DATA)==76833
SEEDS=[20260930,20260931,20260932]
def corr(y,p):
 y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float)
 a=y-y.mean();b=p-p.mean();den=np.sqrt(np.dot(a,a)*np.dot(b,b))
 return float(np.dot(a,b)/den) if den else float('nan')
def metrics(y,p):
 y=np.asarray(y,float);p=np.asarray(p,float);assert np.isfinite(y).all() and np.isfinite(p).all()
 e=y-p;sst=np.sum((y-y.mean())**2)
 return dict(n=len(y),pearson=corr(y,p),spearman=corr(pd.Series(y).rank().to_numpy(),pd.Series(p).rank().to_numpy()),r2=float(1-np.sum(e*e)/sst) if sst else np.nan,rmse=float(np.sqrt(np.mean(e*e))),mae=float(np.mean(abs(e))))
def internal_predictions():
 old=[csvmember(f'phase{t}_predictions_validation_test.csv').assign(protocol=t) for t in ['9b','9c']]
 return pd.concat(old+[pd.read_csv(P12/'results/phase12_internal_predictions_new_arms.csv')],ignore_index=True)
print('Environment:',sys.executable,'; full retained records:',len(DATA))

Environment: C:\Users\User\.conda\envs\enhancer\python.exe ; full retained records: 76833


## 1. Extract local documents and index exact evidence files

In [2]:
import xml.etree.ElementTree as ET
NS={'w':'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
def paragraphs(path):
 with zipfile.ZipFile(path) as z:tree=ET.fromstring(z.read('word/document.xml'))
 return [''.join(p.itertext()) for p in tree.findall('.//w:p',NS)]
# Extract only w:t text to avoid double counting Word's formatting XML.
def paragraphs(path):
 with zipfile.ZipFile(path) as z:tree=ET.fromstring(z.read('word/document.xml'))
 return [''.join(t.text or '' for t in p.findall('.//w:t',NS)) for p in tree.findall('.//w:p',NS)]
review_path=ROOT/'reference_inputs/Review From Journal.docx';manuscript_path=ROOT/'reference_inputs/Submitted Manuscript.docx'
review=paragraphs(review_path);manuscript=paragraphs(manuscript_path)
assert any('ARRAY-D-26-02813' in t for t in review)
for title,para in [('review_letter',review),('submitted_manuscript',manuscript)]:
 (OUT/f'{title}_indexed_paragraphs.txt').write_text('\n'.join(f'P{i+1:04d}: {t}' for i,t in enumerate(para))+'\n',encoding='utf-8')
def summary(number,name):
 p=ROOT/f'phase{number}_outputs'/name
 return json.loads(p.read_text()) if p.exists() else {'status':'pending','reason':'Run notebook '+str(number)+' first'}
verified=summary(13,'verification_summary.json');filtering=summary(14,'filtering_analysis_summary.json')
print('Phase 13:',verified['status'],'; Phase 14:',filtering['status'])
review_text='\n'.join(review)
for name in ['Reviewer #1','Reviewer #4','Reviewer #7']:assert name in review_text

Phase 13: completed ; Phase 14: completed


## 2. Map each scientific reviewer concern to completed evidence and remaining work
New MPRA regression evidence does not retroactively authenticate old independent classification tables. Means/SD across seeds and sampling CIs answer different questions. Status below explicitly distinguishes a technical analysis from manuscript repair.

In [3]:
V='phase13_outputs/verification_summary.json';I='phase13_outputs/internal_metrics_recomputed.csv';X='phase13_outputs/external_metrics_recomputed.csv';B='phase13_outputs/independent_paired_bootstrap.csv';S='phase13_outputs/test_seed_variability.csv';F='phase14_outputs/filtering_analysis_summary.json';D='phase14_outputs/retained_removed_distribution_summary.csv';A='phase14_outputs/cell_type_filter_attrition.csv';G='phase14_outputs/retained_test_subgroup_metrics.csv';P='phase12_outputs/enhancer_phase12_work/results/phase12_provenance.json';N='[12] Enhancer_Kaggle_Phase12_Strong_Baselines.ipynb'
rows=[]
def add(id,concern,status,evidence,finding,remaining,anchor):
 anchors=[i+1 for i,t in enumerate(review) if anchor.lower() in t.lower()]
 assert anchors,(id,anchor)
 rows.append(dict(concern_id=id,concern=concern,status=status,evidence_files=' | '.join(evidence),finding=finding,remaining_action=remaining,review_paragraphs=','.join('P'+str(i).zfill(4) for i in anchors)))
add('R1.1','200 bp/6-mer text versus 2000 bp/4-mer pseudocode','partial',[P,N],'Current MPRA comparison uses 200 bp; one-hot CNNs and DNABERT-2 are separate from the original DNABERT 6-mer model.','Rewrite original pseudocode, task definitions, model names and tokenizer descriptions to match each actual experiment.','Algorithm 1 refers')
add('R1.2','Metric consistency, denominators and confusion matrices','partial',[V,I,X],'New continuous regression metrics are independently recomputed on identical records; confusion matrices are not applicable to these regression targets.','Recover predictions and confusion matrices for every retained historical classification result or remove unauthenticated tables; reconcile actual class counts.','confusion matrices')
add('R1.3','Missing mean/SD and single-run versus repeated results','partial',[S,I,B],'New seed-level scores, descriptive seed SD and three-seed ensemble scores are separated from conditional bootstrap CIs.','Revise tables and legends; old classification tables still need their own run-level accounting.','mean and standard deviation')
add('R1.4','Hyperparameter and epoch selection','partial',[P,'phase12_outputs/enhancer_phase12_work/results/phase12_validation_choices.csv',V],'Phase 12 records repository defaults, train-only target scaling and validation-only epoch maxima; 24 jobs authenticated.','Explain all older experiment searches and selection rules; disclose asymmetric model recipes and no exhaustive tuning.','hyperparameter selection')
add('R1.5','Relevant literature and suggested papers','open',['reference_inputs/Review From Journal.docx'],'Local experimental results cannot establish literature completeness.','Read the suggested primary papers and current MPRA/CNN/genomic foundation-model literature; correct citations based on relevance.','provided references')
add('R1.6','Typos','open',['reference_inputs/Submitted Manuscript.docx'],'New analysis notebooks do not repair the submitted manuscript.','Proofread the revised manuscript, tables, pseudocode and captions.','6=')
add('R4.1','Novelty and incomplete recent research','partial',[I,X,B,F],'Credible contribution is a reproducible evaluation and selection-bias audit; both specialist CNNs beat DNABERT-2 in this tested setting.','Reframe novelty as evidence, not a new architecture or universal transformer advantage; complete literature comparison.','innovation of this work')
add('R4.2','Sequence-insufficiency/generalizability conclusion is unsupported','partial',[I,X,B,F,D,G],'Strong sequence-only CNN results contradict claiming that these experiments prove sequence intrinsically insufficient. Filtering narrows the target population.','Replace sufficiency/insufficiency proof with bounded empirical claims; removed-sequence performance, additional cells and native context remain unknown.','information in the sequence')
add('R4.3','Longformer long-range claims from only 200 bp; suggested 1–2 kb context','open',[P],'Phase 12 tests 200 bp sequence only; external 200 bp transfer is not a flank-length experiment.','Narrow long-context claims or design a separate future flank study; no extra training launched here.','more flanking context')
add('R4.4','Mutation and k-mer shuffling label noise and reproducibility','partial',[N,P],'Current Phase 12 uses random reverse complement in training and forward/RC averaging in inference. It does not validate earlier mutation or shuffle claims.','Inspect older implemented augmentation and revise prose; remove unimplemented operations, or document their label-preservation limits.','limited base substitution')
add('R4.5','Architecture diagrams and equation layout','open',['reference_inputs/Submitted Manuscript.docx'],'Analytical evidence does not resolve figure and equation formatting.','Redraw architecture/pipeline diagrams and standardize equations after final scope is agreed.','messy layout')
add('R7.1','Recalculate Tables 8/14 and reconcile arithmetic','partial',[V,I,X],'Current saved regression metrics reconcile independently; no claim that historical Tables 8/14 have been repaired.','Recalculate historical classification tables from original predictions; do not copy reviewer arithmetic without verifying actual denominators.','Table 8 and Table')
add('R7.2','Algorithm mismatches actual pipeline','partial',[N,P],'Current data length, strand policy and training recipes are traceable.','Replace obsolete pseudocode and make all main-text/table/diagram settings consistent.','Mismatched algorithm')
add('R7.3','Comparative ROC curves, unreadable figures and conflicting settings','partial',[I,X,S,P],'New comparisons are paired regression results with seed diagnostics; ROC curves belong to the old classification task.','Provide appropriate comparative figures for each task, readable labels and consistent scheduler/epoch settings throughout.','Non-standard and problematic')
add('R7.4','Incorrect DNABERT-2 citation and duplicates','open',['reference_inputs/Review From Journal.docx','reference_inputs/Submitted Manuscript.docx'],'No bibliography edits were made.','Check original DNABERT and DNABERT-2 primary citations and remove duplicate Hu entries.','Non-standard reference citations')
add('R7.5','Spelling and typographical errors','open',['reference_inputs/Submitted Manuscript.docx'],'Original manuscript remains preserved.','Comprehensive language and notation pass.','Widespread spelling')
add('Editor.novelty_limits','Explain novelty and limitations','partial',[V,F,B],'Comparison reverses DNABERT-2 superiority; filter audit defines narrower population, uncertainty is conditional.','Rewrite title/abstract/discussion to reflect final evidence and disclose remaining limitations.','novel aspects and limitations')
add('Editor.graphical_abstract','Graphical abstract and highlights','open',['reference_inputs/Review From Journal.docx'],'Analysis notebooks are not publication graphics or highlights.','Prepare graphics and 3–5 accurate highlights after final evidence review.','graphical abstract is required')
add('Editor.submission_materials','Statements, rebuttal, clean/highlighted files and cover letter','open',['reference_inputs/Review From Journal.docx'],'No journal submission or manuscript revision was made. Letter date is historical; current portal status is unverified.','Authors prepare declarations and submission files, and check current editorial status directly.','Data availability statement')
add('Shafin.result_verification','Integrity, record alignment, metrics, ensembles and conditional uncertainty','addressed' if verified.get('status')=='completed' else 'open',[V,I,X,B,S],'Completed CPU verification if summary status is completed; no model retraining. Old Phase 9C K562 collapsed seed retained and diagnosed, root cause unresolved.','Carry verified tables and limitations into revised manuscript; investigate optimizer cause only in a separately authorized future experiment.','systematic validation')
add('Shafin.filter_population','Retained-versus-removed population and error subgroups','addressed' if filtering.get('status')=='completed' else 'partial',[F,D,A,G],'Raw population reconstructed and retained records matched exactly if Phase 14 summary is completed; removed-sequence prediction performance remains unavailable.','Report selection effects and bounded population claims; do not impute removed-sequence accuracy.','generalizability of the results')
map_df=pd.DataFrame(rows)
# A missing evidence file always downgrades an addressed row and is explicitly listed.
map_df['missing_evidence']=map_df.evidence_files.map(lambda s:' | '.join(p for p in s.split(' | ') if not (ROOT/p).is_file()))
map_df.loc[map_df.missing_evidence.ne('')&map_df.status.eq('addressed'),'status']='partial'
write_table('reviewer_evidence_map.csv',map_df)
print(map_df[['concern_id','status','concern','missing_evidence']].to_string(index=False))

                 concern_id    status                                                                     concern missing_evidence
                       R1.1   partial                           200 bp/6-mer text versus 2000 bp/4-mer pseudocode                 
                       R1.2   partial                     Metric consistency, denominators and confusion matrices                 
                       R1.3   partial                      Missing mean/SD and single-run versus repeated results                 
                       R1.4   partial                                          Hyperparameter and epoch selection                 
                       R1.5      open                                    Relevant literature and suggested papers                 
                       R1.6      open                                                                       Typos                 
                       R4.1   partial                                      Novelty 

## 3. Claims requiring narrower wording
The suggested statements below are evidence-bound drafts for discussion. They must be integrated with the actual revised manuscript; this notebook does not assume an unpublished revised manuscript exists.

In [4]:
claims=[
 ('DNABERT-2 outperforms strong specialist CNNs.','Under the tested 200 bp recipes and frozen splits, MPRA-LegNet and MPRAnn ensembles outperform DNABERT-2 on within-assay Pearson and DHS-natural transfer Spearman.',I+' | '+X),
 ('Primary sequence is insufficient for robust enhancer strength prediction.','Sequence-only models predict part of the measured activity variation in these assays. These experiments do not establish an intrinsic information limit or the benefit of adding epigenomic context.',I),
 ('These results generalize broadly to enhancers.','Results concern filtered 200 bp assayed candidate elements from two cell/design families, with external transfer examined separately. Performance on sequences removed by the legacy-similarity screen is unknown.',F+' | '+A+' | '+X),
 ('The confidence intervals express total model uncertainty.','Paired bootstrap intervals are conditional on trained models and previously examined partitions. Seed variability is reported separately; sequence-wise external resampling is supplemented by a 1 kb region sensitivity analysis.',B+' | '+S),
 ('Longformer captures useful long-range regulatory context here.','The experiments using 200 bp inputs do not test the value of kilobase-scale flanking context.',P),
 ('The old failed seed was fixed.','The earlier Phase 9C K562 DNABERT-2 seed remains nearly constant and is included in the archived three-seed ensemble; the cause is unresolved, with healthy-seed sensitivity reported separately.','phase13_outputs/old_9c_k562_collapse_sensitivity.csv')]
claim_df=pd.DataFrame(claims,columns=['claim_to_avoid','suggested_bounded_wording','evidence_files']);write_table('manuscript_claim_revisions.csv',claim_df)
index=[]
for rel in sorted(set([p for s in map_df.evidence_files for p in s.split(' | ')]+['reference_inputs/Review From Journal.docx','reference_inputs/Submitted Manuscript.docx'])):
 p=ROOT/rel;index.append(dict(relative_path=rel,absolute_path=str(p),exists=p.is_file(),sha256=filehash(p) if p.is_file() else '',bytes=p.stat().st_size if p.is_file() else 0))
write_table('evidence_file_index.csv',pd.DataFrame(index))
lines=['# Reviewer evidence map — Shafin, 2026-10-12','','Status meanings: addressed = technical evidence completed; partial = evidence helps but scientific/manuscript work remains; open = not resolved. No editorial acceptance is implied.','']
for row in map_df.itertuples(index=False):lines.extend([f'## {row.concern_id} — {row.status}',row.concern,'',row.finding,'Remaining: '+row.remaining_action,'Review source paragraphs: '+row.review_paragraphs,'Evidence: '+row.evidence_files,''])
lines.extend(['## Suggested claim revisions',''])
for old,new,ev in claims:lines.extend(['Avoid: '+old,'Suggested: '+new,'Evidence: '+ev,''])
(OUT/'reviewer_evidence_map.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
dump_json('reviewer_map_summary.json',dict(status='completed',phase13_status=verified['status'],phase14_status=filtering['status'],concerns=len(map_df),counts=map_df.status.value_counts().to_dict(),missing_evidence_rows=int(map_df.missing_evidence.ne('').sum()),manuscript_edited=False,journal_contacted=False))
print('Evidence map saved. Remaining scientific and manuscript issues are explicitly open/partial.')

Evidence map saved. Remaining scientific and manuscript issues are explicitly open/partial.
